In [ ]:
import json
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import cohen_kappa_score, make_scorer
from tqdm import tqdm
import xgboost as xgb

import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.svm import LinearSVR
from scipy.optimize import minimize


In [ ]:
class OrderedLogitWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, method: str = "lbfgs", maxiter: int = 200):
        self.method = method
        self.maxiter = maxiter

    def _to_matrix(self, X):
        if hasattr(X, "toarray"):
            return X.toarray()
        return np.asarray(X)

    def fit(self, X, y):
        X_mat = self._to_matrix(X)
        self.model_ = OrderedModel(y, X_mat, distr="logit")
        self.result_ = self.model_.fit(method=self.method, maxiter=self.maxiter, disp=False)
        self.classes_ = np.sort(np.unique(y))
        return self

    def predict_proba(self, X):
        X_mat = self._to_matrix(X)
        probs = self.result_.model.predict(self.result_.params, exog=X_mat)
        return probs

    def predict(self, X):
        return self.predict_proba(X).argmax(1)



In [30]:
DATA_DIR = Path(".")
TRAIN_PATH = DATA_DIR / "../data/train.csv"
TEST_PATH = DATA_DIR / "../data/test.csv"
MODEL_PATH = DATA_DIR / "../models/ordinal_model.pkl"
CV_JSON = DATA_DIR / "../data/cv_scores.json"
PRED_PATH = DATA_DIR / "../data/predictions.csv"

In [31]:
train_df = pd.read_csv(TRAIN_PATH, header=0, skiprows=lambda x: 0 < x < 21)
test_df = pd.read_csv(TEST_PATH)

train_df.columns = train_df.columns.str.strip()
if 'id' in train_df.columns:
    train_df = train_df.drop(columns=['id'])
test_df.columns = test_df.columns.str.strip()
if 'id' in test_df.columns:
    test_df = test_df.drop(columns=['id'])
    
categorical_cols = train_df.select_dtypes(include=['object']).columns
train_df = pd.get_dummies(train_df, columns=categorical_cols)
train_df.columns = train_df.columns.str.strip()

categorical_cols = test_df.select_dtypes(include=['object']).columns
test_df = pd.get_dummies(test_df, columns=categorical_cols)
test_df.columns = test_df.columns.str.strip()

print("Train shape:", train_df.shape)
print("Test  shape:", test_df.shape)

Train shape: (3940, 114)
Test  shape: (20, 83)


In [ ]:
TARGET = "sii"

train_df = train_df.dropna(subset=[TARGET]).copy()
train_df[TARGET] = train_df[TARGET].astype(int)

X = train_df.drop(columns=[TARGET])
y = train_df[TARGET]

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()

print(f"Categorical: {len(cat_cols)} | Numerical: {len(num_cols)}")


Categorical: 0 | Numerical: 113


In [33]:
preprocess = ColumnTransformer([
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols),
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), num_cols)
])


In [34]:
def qwk_score(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")

qwk_scorer = make_scorer(qwk_score)


In [ ]:
model = Pipeline([
    ("prep", preprocess),
    ("clf", OrderedLogitWrapper(
            method="lbfgs",
            maxiter=200))
])



In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=skf, scoring=qwk_scorer, n_jobs=-1)

print("CV QWK scores:", cv_scores)
print("Mean QWK:", cv_scores.mean().round(4))

CV_JSON.write_text(json.dumps({"fold_scores": cv_scores.tolist(), "mean": cv_scores.mean()}, indent=2))

CV QWK scores: [0.97223417 0.98458946 0.98306952 0.98916467 0.98768389]
Mean QWK: 0.9833


175

In [ ]:
model.fit(X, y)
joblib.dump(model, MODEL_PATH)
print("Model salvat la", MODEL_PATH)

Model salvat la ..\models\ordinal_model.pkl


c:\Program Files\Python312\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


In [39]:
missing_columns = set(train_df.columns) - set(test_df.columns)
missing_columns.remove('sii')

print("Coloanele care lipsesc:", missing_columns)

for col in missing_columns:
    print(f"\nPrezicem coloana: {col}")
    
    X_train_missing = train_df.drop(columns=list(missing_columns) + ['sii'])
    y_train_missing = train_df[col]

    if y_train_missing.isna().any():
        print(f"Atenție: {y_train_missing.isna().sum()} valori NaN în coloana țintă")
        
        print("Vom prezice valorile NaN din coloana țintă")
        
        mask_known = ~y_train_missing.isna()
        X_known = X_train_missing[mask_known]
        y_known = y_train_missing[mask_known]
        
        X_unknown = X_train_missing[~mask_known]
        
        nan_model = xgb.XGBRegressor(
            max_depth=5,
            learning_rate=0.1,
            n_estimators=100,
            colsample_bytree=0.8,
            min_child_weight=1,
            gamma=0.1,
            subsample=0.8,
            random_state=42
        )
        nan_model.fit(X_known, y_known)
        
        y_pred_nan = nan_model.predict(X_unknown)
        
        y_train_missing.loc[~mask_known] = y_pred_nan
        
        print(f"Am completat {(~mask_known).sum()} valori NaN în coloana țintă")
    
    if not np.isfinite(y_train_missing).all():
        print("Atenție: Există valori infinite în coloana țintă")
        
        median_value = np.median(y_train_missing[np.isfinite(y_train_missing)])
        y_train_missing = y_train_missing.replace([np.inf, -np.inf], median_value)
        
        print(f"Am înlocuit valorile infinite cu mediana: {median_value}")

    missing_model = xgb.XGBRegressor(
        max_depth=5,
        learning_rate=0.1,
        n_estimators=100,
        colsample_bytree=0.8,
        min_child_weight=1,
        gamma=0.1,
        subsample=0.8,
        random_state=42
    )
    missing_model.fit(X_train_missing, y_train_missing)
    
    X_test_missing = test_df.drop(columns=[col for col in missing_columns if col in test_df.columns])
    test_df[col] = missing_model.predict(X_test_missing)
    
    train_score = missing_model.score(X_train_missing, y_train_missing)
    print(f"Scor pe setul de antrenament: {train_score:.4f}")

Coloanele care lipsesc: {'PCIAT-PCIAT_09', 'PCIAT-Season_Spring', 'PAQ_A-Season_Fall', 'PCIAT-PCIAT_07', 'PCIAT-PCIAT_06', 'PCIAT-PCIAT_04', 'PCIAT-PCIAT_15', 'PCIAT-Season_Fall', 'PCIAT-PCIAT_18', 'PCIAT-PCIAT_12', 'PCIAT-PCIAT_08', 'PCIAT-PCIAT_02', 'PAQ_A-Season_Winter', 'PCIAT-PCIAT_03', 'PCIAT-Season_Summer', 'PCIAT-PCIAT_10', 'PAQ_A-Season_Spring', 'PCIAT-PCIAT_01', 'PCIAT-PCIAT_20', 'BIA-Season_Spring', 'PCIAT-Season_Winter', 'PCIAT-PCIAT_05', 'PCIAT-PCIAT_14', 'PCIAT-PCIAT_17', 'PCIAT-PCIAT_Total', 'PCIAT-PCIAT_16', 'PCIAT-PCIAT_19', 'PCIAT-PCIAT_11', 'PCIAT-PCIAT_13', 'Fitness_Endurance-Season_Winter'}

Prezicem coloana: PCIAT-PCIAT_09
Atenție: 6 valori NaN în coloana țintă
Vom prezice valorile NaN din coloana țintă


C:\Users\stefa\AppData\Local\Temp\ipykernel_20016\2330823834.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y_train_missing.loc[~mask_known] = y_pred_nan


Am completat 6 valori NaN în coloana țintă
Scor pe setul de antrenament: 0.7040

Prezicem coloana: PCIAT-Season_Spring
Scor pe setul de antrenament: 0.9187

Prezicem coloana: PAQ_A-Season_Fall
Scor pe setul de antrenament: 0.8879

Prezicem coloana: PCIAT-PCIAT_07
Atenție: 7 valori NaN în coloana țintă
Vom prezice valorile NaN din coloana țintă
Am completat 7 valori NaN în coloana țintă


C:\Users\stefa\AppData\Local\Temp\ipykernel_20016\2330823834.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y_train_missing.loc[~mask_known] = y_pred_nan


Scor pe setul de antrenament: 0.7878

Prezicem coloana: PCIAT-PCIAT_06
Atenție: 4 valori NaN în coloana țintă
Vom prezice valorile NaN din coloana țintă
Am completat 4 valori NaN în coloana țintă


...

Am completat 6 valori NaN în coloana țintă
Scor pe setul de antrenament: 0.6868

Prezicem coloana: PCIAT-PCIAT_11
Atenție: 2 valori NaN în coloana țintă
Vom prezice valorile NaN din coloana țintă


C:\Users\stefa\AppData\Local\Temp\ipykernel_20016\2330823834.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y_train_missing.loc[~mask_known] = y_pred_nan


Am completat 2 valori NaN în coloana țintă
Scor pe setul de antrenament: 0.6953

Prezicem coloana: PCIAT-PCIAT_13
Atenție: 7 valori NaN în coloana țintă
Vom prezice valorile NaN din coloana țintă


C:\Users\stefa\AppData\Local\Temp\ipykernel_20016\2330823834.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y_train_missing.loc[~mask_known] = y_pred_nan


Am completat 7 valori NaN în coloana țintă
Scor pe setul de antrenament: 0.6396

Prezicem coloana: Fitness_Endurance-Season_Winter
Scor pe setul de antrenament: 0.9538


In [ ]:
y_test_pred = model.predict(test_df)

if "id" in test_df.columns:
    out_df = pd.DataFrame({"id": test_df["id"], TARGET + "_pred": y_test_pred})
else:
    out_df = pd.DataFrame({"id": np.arange(len(test_df)), TARGET + "_pred": y_test_pred})

out_df.to_csv(PRED_PATH, index=False)
print("Predicții salvate la", PRED_PATH)


Predicții salvate la ..\data\predictions.csv
